In [2]:
pip install tpot

  Using cached TPOT-0.12.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached update_checker-0.18.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached stopit-1.1.2.tar.gz (18 kB)
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
Using cached TPOT-0.12.2-py3-none-any.whl (87 kB)
   ---------------------------------------- 0.0/109.8 kB ? eta -:--:--
   --- ------------------------------------ 10.2/109.8 kB ? eta -:--:--
   ---------------------------------------- 109.8/109.8 kB 1.6 MB/s eta 0:00:00
Using cached update_checker-0.18.0-py3-none-any.whl (7.0 kB)
   ---------------------------------------- 0.0/150.0 MB ? eta -:--:--
   ---------------------------------------- 0.8/150.0 MB 26.2 MB/s eta 0:00:06
    --------------------------------------- 2.0/150.0 MB 42.3 MB/s eta 0:00:04
   - -------------------------------------- 5.2/150.0 MB 41.4 MB/s eta 0:00:04
   -- ------------------------------------- 7.9/150.0 MB 46.1 MB/s eta 0:00:04


In [1]:
import pandas as pd
import numpy as np
from dateutil.relativedelta import relativedelta
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tpot import TPOTClassifier
from sklearn.metrics import roc_auc_score, roc_curve, auc
import importlib.util
import matplotlib.pyplot as plt

## 전처리 자동화 함수

In [2]:
def preprocess(df, label_encoders=None, fit_encoders=True):
    # 날짜 컬럼을 datetime 형식으로 변환하고 파생 변수 생성
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
    df['dob'] = pd.to_datetime(df['dob'])

    df['trans_hour'] = df['trans_date_trans_time'].dt.hour
    df['trans_minute'] = df['trans_date_trans_time'].dt.minute
    df['trans_day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
    df['trans_year_month'] = df['trans_date_trans_time'].dt.to_period('M').astype(str)

    # 나이 계산 함수 정의
    def calculate_age(born, trans_date):
        return relativedelta(trans_date, born).years

    df['age'] = df.apply(lambda x: calculate_age(x['dob'], x['trans_date_trans_time']), axis=1)

    # 불필요한 컬럼 삭제
    df = df.drop(['trans_date_trans_time', 'dob', 'first', 'last', 'trans_num'], axis=1, errors='ignore')

    # 새로운 컬럼 생성 (merchant와 category 합침)
    df['merchant_category'] = df['merchant'].fillna('') + ':' + df['category'].fillna('')

    # 라벨 인코딩할 컬럼 목록
    label_cols = ['merchant', 'category', 'gender', 'street', 'city', 'state',
                  'job', 'trans_year_month', 'merchant_category']

    if label_encoders is None:
        label_encoders = {}

    for col in label_cols:
        if fit_encoders:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            label_encoders[col] = le
        else:
            le = label_encoders[col]
            # 학습 시 존재하지 않은 카테고리는 'unknown'으로 대체
            df[col] = df[col].astype(str).map(lambda x: x if x in le.classes_ else 'unknown')
            if 'unknown' not in le.classes_:
                le.classes_ = np.append(le.classes_, 'unknown')
            df[col] = le.transform(df[col])

    return df, label_encoders

## TPOT 학습중 선택한 model, feature 확인

In [3]:
def pipeline_str_to_object(pipeline_str):
    """
    TPOT가 생성한 pipeline 코드를 실행하여 pipeline 객체로 변환
    """
    local_vars = {}
    exec(pipeline_str, globals(), local_vars)
    return local_vars['pipeline']

def get_selected_features(pipeline, feature_names):
    """
    pipeline 내에 SelectKBest가 있다면 선택된 feature 이름 리스트 반환
    """
    selected_features = None
    if 'selectkbest' in pipeline.named_steps:
        selector = pipeline.named_steps['selectkbest']
        selected_indices = selector.get_support(indices=True)
        selected_features = feature_names[selected_indices]
    return selected_features

def save_tpot_evolution_log_with_features(tpot, feature_names, filename="tpot_evolution_log_with_features.txt"):
    """
    TPOT 진화 과정 각 개체별 pipeline, 성능, 선택된 feature 목록을 파일로 저장
    """
    with open(filename, "w", encoding="utf-8") as f:
        f.write("TPOT 진화 과정 로그 (선택된 피처 포함)\n")
        f.write("===================================\n\n")

        for gen_idx, generation in enumerate(tpot.evolutionary_process_):
            f.write(f"세대 {gen_idx + 1}:\n")
            for ind_idx, individual in enumerate(generation):
                pipeline_str = individual['pipeline_str']
                internal_cv_score = individual['internal_cv_score']

                # pipeline 객체로 변환 시도
                try:
                    pipeline = pipeline_str_to_object(pipeline_str)
                except Exception as e:
                    pipeline = None
                    f.write(f"  개체 {ind_idx + 1}:\n")
                    f.write(f"    ROC AUC 점수: {internal_cv_score:.4f}\n")
                    f.write(f"    파이프라인 변환 실패: {e}\n\n")
                    continue

                f.write(f"  개체 {ind_idx + 1}:\n")
                f.write(f"    ROC AUC 점수: {internal_cv_score:.4f}\n")
                f.write(f"    파이프라인 코드:\n{pipeline_str}\n")

                selected_features = get_selected_features(pipeline, feature_names)
                if selected_features is not None:
                    f.write("    * 선택된 피처 목록:\n")
                    for feat in selected_features:
                        f.write(f"      - {feat}\n")
                else:
                    f.write("    * 명시적인 피처 선택 단계 없음\n")
                f.write("\n")
            f.write("\n")
    print(f"\n TPOT 진화 과정 + 선택된 피처 로그가 '{filename}'에 저장되었습니다.")

In [4]:
# 1. 데이터 불러오기
df_train = pd.read_csv('./data/fraudTrain.csv', index_col=0)
df_test = pd.read_csv('./data/fraudTest.csv', index_col=0)

In [5]:
# 2. 전처리 - 학습 데이터 (라벨 인코더 생성 및 학습)
df_train_processed, label_encoders = preprocess(df_train, label_encoders=None, fit_encoders=True)

# 3. 전처리 - 테스트 데이터 (기존 라벨 인코더 사용)
df_test_processed, _ = preprocess(df_test, label_encoders=label_encoders, fit_encoders=False)

In [6]:
# 4. 피처와 타겟 분리
target = 'is_fraud'

X_train = df_train_processed.drop(columns=[target])
y_train = df_train_processed[target]

X_test = df_test_processed.drop(columns=[target])
y_test = df_test_processed[target]

feature_names = X_train.columns

In [13]:
# 5. 학습용 데이터에서 검증용 데이터 분리 (옵션)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

In [14]:
X_tr

,cc_num,merchant,category,amt,gender,street,city,state,zip,lat,...,job,unix_time,merch_lat,merch_long,trans_hour,trans_minute,trans_day_of_week,trans_year_month,age,merchant_category
509059,639023984367,600,3,51.71,0,799,322,34,13647,44.6087,...,74,1344482007,44.785928,-74.659301,3,13,4,7,28,607
395295,373905417449658,151,0,13.78,0,557,501,43,76665,31.9290,...,340,1340999808,31.414028,-98.152203,19,56,5,5,48,154
536531,3553629419254918,341,12,961.26,0,717,171,47,98238,48.3400,...,390,1345300940,49.118546,-122.622065,14,42,6,7,34,346
271001,371034293500716,493,5,43.68,1,581,831,4,96135,40.0235,...,143,1336944466,39.528098,-121.059990,21,27,0,4,53,499
532788,4335531783520911,644,0,33.08,0,845,615,24,65066,38.3511,...,308,1345212941,39.213785,-92.188153,14,15,5,7,21,651
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125527,30044330818990,152,0,19.70,0,872,271,9,33967,26.4722,...,327,1331413575,26.986105,-82.688123,21,6,6,2,33,155
150047,3587044315915002,467,12,123.45,1,505,295,42,38039,35.0659,...,91,1332392799,35.018147,-88.395919,5,6,4,2,44,473
1151888,4169759661243568,2,1,64.98,0,554,434,38,17041,40.2236,...,428,1366913512,40.191038,-77.350918,18,11,5,15,48,2
547306,30118423745458,671,7,33.65,1,12,485,31,7747,40.4109,...,136,1345647578,40.081094,-73.434345,14,59,3,7,26,678


In [15]:
X_tr = np.array(X_tr, copy=True)
X_val = np.array(X_val, copy=True)
y_tr = np.array(y_tr, copy=True)
y_val = np.array(y_val, copy=True)

In [16]:
X_tr

array([[6.39023984e+11, 6.00000000e+02, 3.00000000e+00, ...,
        7.00000000e+00, 2.80000000e+01, 6.07000000e+02],
       [3.73905417e+14, 1.51000000e+02, 0.00000000e+00, ...,
        5.00000000e+00, 4.80000000e+01, 1.54000000e+02],
       [3.55362942e+15, 3.41000000e+02, 1.20000000e+01, ...,
        7.00000000e+00, 3.40000000e+01, 3.46000000e+02],
       ...,
       [4.16975966e+15, 2.00000000e+00, 1.00000000e+00, ...,
        1.50000000e+01, 4.80000000e+01, 2.00000000e+00],
       [3.01184237e+13, 6.71000000e+02, 7.00000000e+00, ...,
        7.00000000e+00, 2.60000000e+01, 6.78000000e+02],
       [2.71220973e+15, 1.61000000e+02, 1.00000000e+01, ...,
        0.00000000e+00, 4.10000000e+01, 1.64000000e+02]])

In [ ]:
# 6. TPOT 모델 학습
tpot = TPOTClassifier(
    generations=5,
    population_size=20,
    verbosity=2,
    random_state=42,
    n_jobs=1,
    scoring='roc_auc'
)
tpot.fit(X_tr, y_tr)

Version 0.12.2 of tpot is outdated. Version 1.0.0 was released Wednesday February 26, 2025.


Optimization Progress:   0%|          | 0/120 [00:00<?, ?pipeline/s]

In [ ]:
# 7. 검증용 데이터로 성능 평가
val_score = tpot.score(X_val, y_val)
print(f"\n TPOT 외부 검증 ROC AUC 점수: {val_score:.4f}")

In [ ]:
# 8. 최적 파이프라인 코드 저장
pipeline_path = 'tpot_best_pipeline.py'
tpot.export(pipeline_path)
print(f"\n 최적 파이프라인 코드가 '{pipeline_path}'에 저장되었습니다.")

In [ ]:
# 9. 저장한 최적 파이프라인 불러오기
spec = importlib.util.spec_from_file_location("tpot_pipeline", pipeline_path)
pipeline_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline_module)
pipeline = pipeline_module.exported_pipeline

In [ ]:
# 10. 테스트 데이터 예측 및 성능 평가
y_pred_proba = pipeline.predict_proba(X_test)[:,1]
test_roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"\n 테스트 데이터 ROC AUC 점수: {test_roc_auc:.4f}")

# ROC AUC 점수

**ROC**는 Receiver Operating Characteristic의 약자 
- 분류기 성능 평가를 위한 도구로, 수신자가 신호(Positive)를 잘 받아내는지, 오탐은 얼마나 되는지 균형을 보는 곡선

**AUC**는 Area Under the Curve의 약자로, ROC 곡선 아래 면적을 의미

---

## ROC 곡선

이진 분류 문제(사기 거래 vs 정상 거래)에서 모델이 얼마나 잘 구분하는지 평가하는 그래프 
- X축: False Positive Rate (거짓 양성 비율)  
- Y축: True Positive Rate (참 양성 비율, 즉 민감도)

모델의 임계값을 변화시키며 이 두 값을 계산해 그린 곡선이 ROC 곡선

---

## AUC 점수

ROC 곡선 아래 면적이며 0과 1 사이 값 
- 1에 가까울수록 좋은 모델  
- 0.5는 무작위 추측 수준
- 1은 완벽한 분류를 의미

In [ ]:
# 10-1. ROC 곡선 시각화
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8,6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC 곡선 (면적 = {roc_auc:.4f})')
plt.plot([0,1], [0,1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('거짓 양성 비율 (False Positive Rate)')
plt.ylabel('참 양성 비율 (True Positive Rate)')
plt.title('TPOT 모델 테스트 데이터 ROC 곡선')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

In [ ]:
# 11. 최종 파이프라인에서 선택된 피처 출력
selected_features_final = get_selected_features(pipeline, feature_names)
if selected_features_final is not None:
    print("\n 최종 파이프라인에서 선택된 피처 목록:")
    for feat in selected_features_final:
        print(f"  - {feat}")
else:
    print("\n 최종 파이프라인에 명시적인 피처 선택 단계가 없습니다.")

In [ ]:
# 12. TPOT 진화 과정 로그 및 선택된 피처 목록 파일로 저장
save_tpot_evolution_log_with_features(tpot, feature_names)